# Data Preparation — RELAX Dataset

This notebook processes raw per-week CSV exports (InfluxDB format) into
per-participant Parquet files for publication.

**Pipeline:**
1. Parse per-week, per-signal CSV files and merge into a single ACC and IBI
   DataFrame per participant.
2. Concatenate all weeks and save as intermediate CSV files.
3. Convert intermediate CSVs to Apache Parquet with corrected column types.

**Input:** `raw_data/{intervention}_{participant_id}_{week}_{signal}.csv`  
**Output:** `RELAXDataset/data/{participant_id}/acc_data.parquet` and `ibi_data.parquet`


In [ ]:
import os
import pandas as pd
from pathlib import Path
from fastparquet import write

DATA_PATH = "raw_data/"
OUTPUT_PATH = "Final/data/"

os.listdir(DATA_PATH)
participant_intervention_dict = {}
for file in os.listdir(DATA_PATH):
    file_name_list = file.split("_")
    participant_intervention_dict[file_name_list[1]] = file_name_list[0]


## Helper Functions

`get_whole_data_of_week` extends the base ACC and IBI DataFrames with the remaining
signals (`acc_y`, `acc_z`, `ibi_blocker`, `ibi_errorEstimate`) for a given participant
and week. Files are joined on the timestamp index; missing or mismatched files are
skipped with a warning.


In [3]:
def get_whole_data_of_week(participant_id,week, data_acc, data_ibi):
    intervention_type = participant_intervention_dict[participant_id]
    features_acc = ['acc_y', 'acc_z']
    features_ibi = ['ibi_blocker', 'ibi_errorEstimate']

    for feature in features_acc:
        try:
            data_name = f"{intervention_type}_{participant_id}_{week}_{feature}.csv"
            data_acc_new = pd.read_csv(DATA_PATH + data_name, skiprows=3, index_col="_time",usecols=["_time", "_value"],dtype={'_time': 'str', '_value': "float64"},parse_dates=["_time"])
            data_acc_new.rename(columns={"_value": f"{feature}"}, inplace=True)
            if data_acc_new.index.equals(data_acc.index):
                data_acc = data_acc.join(data_acc_new, how="inner")
            else:
                print(f"Index mismatch for {data_name}. Skipping this file.")
        except FileNotFoundError:
            print(f"File {data_name} not found. Skipping this file.")
    
    for feature in features_ibi: 
        try:
            data_name = f"{intervention_type}_{participant_id}_{week}_{feature}.csv"
            data_ibi_new = pd.read_csv(DATA_PATH + data_name, skiprows=3, index_col="_time",usecols=["_time", "_value"],dtype={'_time': 'str', '_value': "float64"},parse_dates=["_time"])
            data_ibi_new.rename(columns={"_value": f"{feature}"}, inplace=True)
            if data_ibi_new.index.equals(data_ibi.index):
                data_ibi = data_ibi.join(data_ibi_new, how="inner")
        except FileNotFoundError:
            print(f"File {data_name} not found. Skipping this file.")
    return data_acc, data_ibi

## Known Data Issues

| Participant | Issue |
|-------------|-------|
| 57 | IBI files for week 1 missing — week skipped |
| 27 | Week 3 files present but empty — week skipped |
| 63 | Weeks 0 and 3 files present but empty — weeks skipped |


## Processing Pipeline

- `read_timeseries_csv` — reads a single signal CSV (3-row InfluxDB header) into a
  timestamped DataFrame.
- `load_week_data` — loads ACC (`acc_x`) and IBI (`ibi_ppi`) for one week, then appends
  remaining signals via `get_whole_data_of_week`.
- `process_participant` — iterates over all weeks, concatenates, and writes
  `acc_data.csv` / `ibi_data.csv` per participant.
- `process_all_participants` — runs the pipeline for all participants, skipping those
  already processed.


In [ ]:
SKIP_PARTICIPANTS = {}  # extend if necessary


def read_timeseries_csv(path, value_col_name):
    df = pd.read_csv(
        path,
        skiprows=3,
        index_col="_time",
        usecols=["_time", "_value"],
        dtype={"_time": "str", "_value": "float64"},
        parse_dates=["_time"],
    )
    
    return df.rename(columns={"_value": value_col_name})


def load_week_data(participant_id, intervention_type, week, data_path):
    acc_path = Path(data_path) / f"{intervention_type}_{participant_id}_{week}_acc_x.csv"
    ibi_path = Path(data_path) / f"{intervention_type}_{participant_id}_{week}_ibi_ppi.csv"

    acc = read_timeseries_csv(acc_path, "acc_x")
    ibi = read_timeseries_csv(ibi_path, "ibi_ppi")

    acc, ibi = get_whole_data_of_week(participant_id, str(week), acc, ibi)
    return acc, ibi


def process_participant(participant_id, intervention_type, data_path, output_path, weeks=(0, 1, 2, 3)):
    output_path = Path(output_path)
    acc_dir = output_path / "acc" / participant_id
    ibi_dir = output_path / "ibi" / participant_id

    acc_weeks = []
    ibi_weeks = []

    for week in weeks:
        try:
            acc_w, ibi_w = load_week_data(participant_id, intervention_type, week, data_path)
        except FileNotFoundError as e:
            print(f"[{participant_id}] week {week}: missing file -> {e}. Skipping week.")
            continue
        except pd.errors.EmptyDataError as e:
            print(f"[{participant_id}] week {week}: empty CSV -> {e}. Skipping week.")
            continue
        acc_weeks.append(acc_w)
        ibi_weeks.append(ibi_w)

    acc_all = pd.concat(acc_weeks, axis=0)
    ibi_all = pd.concat(ibi_weeks, axis=0)

    print(
        f"Processed participant {participant_id} ({intervention_type}): "
        f"acc={acc_all.shape[0]} rows, ibi={ibi_all.shape[0]} rows."
    )

    acc_dir.mkdir(parents=True, exist_ok=True)
    ibi_dir.mkdir(parents=True, exist_ok=True)

    acc_all.to_csv(acc_dir / "acc_data.csv")
    ibi_all.to_csv(ibi_dir / "ibi_data.csv")

    print(f"Saved data for participant {participant_id}.")


def process_all_participants(participant_intervention_dict, data_path, output_path):
    acc_root = Path(output_path) / "acc"
    existing = set(os.listdir(acc_root)) if acc_root.exists() else set()

    for participant_id, intervention_type in participant_intervention_dict.items():
        if participant_id in existing:
            continue

        if participant_id in SKIP_PARTICIPANTS:
            print(f"Skipping participant {participant_id} due to known data issues.")
            continue

        print(f"Processing participant {participant_id} ({intervention_type})")
        process_participant(participant_id, intervention_type, data_path, output_path)

In [13]:
process_all_participants(participant_intervention_dict, DATA_PATH, OUTPUT_PATH)

Processing participant 63 (global)
[63] week 0: empty CSV -> No columns to parse from file. Skipping week.
[63] week 3: empty CSV -> No columns to parse from file. Skipping week.
Processed participant 63 (global): acc=29288435 rows, ibi=656181 rows.
Saved data for participant 63.
Processing participant 27 (local)
[27] week 3: empty CSV -> No columns to parse from file. Skipping week.
Processed participant 27 (local): acc=69055923 rows, ibi=1631482 rows.
Saved data for participant 27.
Processing participant 57 (local)
[57] week 1: missing file -> [Errno 2] No such file or directory: 'raw_data/local_57_1_ibi_ppi.csv'. Skipping week.
Processed participant 57 (local): acc=63378213 rows, ibi=1315568 rows.
Saved data for participant 57.


## CSV to Parquet Conversion

Intermediate CSVs are converted to Apache Parquet (SNAPPY compression) with corrected
column types:

- `timestamp` — parsed from `_time` (ISO 8601, UTC-aware)
- `ibi_blocker` — cast to `boolean`
- `ibi_errorEstimate`, `ibi_ppi` — cast to `int32`


In [ ]:
def save_parquet(participant_id):
    OUTPUT_PATH = Path("../../RELAXDataset/data/")
    INPUT_PATH = Path("../../Processed_for_OSF/data/")

    dir_acc_in = INPUT_PATH / "acc" / participant_id
    dri_acc_out = OUTPUT_PATH / participant_id
    df = pd.read_csv(dir_acc_in / "acc_data.csv")


    df["timestamp"] = pd.to_datetime(
        df["_time"],
        format="ISO8601",
        utc=True
    )
    df = df.drop(columns=["_time"])

    dri_acc_out.mkdir(parents=True, exist_ok=True)

    write(
        dri_acc_out / "acc_data.parquet",
        df,
        compression="SNAPPY",
        write_index=False
    )


    dir_ibi_in = INPUT_PATH / "ibi" / participant_id
    dir_ibi_out = OUTPUT_PATH / participant_id
    df = pd.read_csv(dir_ibi_in / "ibi_data.csv")

    df["timestamp"] = pd.to_datetime(
        df["_time"],
        format="ISO8601",
        utc=True
    )
    df = df.drop(columns=["_time"])
    df["ibi_blocker"] = df["ibi_blocker"].astype("int").astype("boolean")
    df["ibi_errorEstimate"] = df["ibi_errorEstimate"].astype("int32")
    df["ibi_ppi"] = df["ibi_ppi"].astype("int32")

    write(
        dir_ibi_out / "ibi_data.parquet",
        df,
        compression="SNAPPY",
        write_index=False
    )


In [7]:
i = 1
for P_ID in participant_intervention_dict.keys():
    save_parquet(P_ID)
    print(f"{i}. Saved parquet for participant {P_ID}")
    i += 1

1. Saved parquet for participant 16
2. Saved parquet for participant 45
3. Saved parquet for participant 31
4. Saved parquet for participant 60
5. Saved parquet for participant 56
6. Saved parquet for participant 63
7. Saved parquet for participant 38
8. Saved parquet for participant 48
9. Saved parquet for participant 15
10. Saved parquet for participant 14
11. Saved parquet for participant 24
12. Saved parquet for participant 12
13. Saved parquet for participant 22
14. Saved parquet for participant 46
15. Saved parquet for participant 23
16. Saved parquet for participant 43
17. Saved parquet for participant 27
18. Saved parquet for participant 41
19. Saved parquet for participant 34
20. Saved parquet for participant 49
21. Saved parquet for participant 28
22. Saved parquet for participant 44
23. Saved parquet for participant 18
24. Saved parquet for participant 25
25. Saved parquet for participant 13
26. Saved parquet for participant 58
27. Saved parquet for participant 40
28. Saved 